In [1]:
import os
import urllib.request

os.makedirs("../data", exist_ok=True)

docs = {
    "nist_sp800_177r1.pdf": "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-177r1.pdf",
    "nist_sp800_61r2.pdf":  "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-61r2.pdf",
}

for filename, url in docs.items():
    path = f"../data/{filename}"
    
    if not os.path.exists(path):
        print(f"Downloading {filename}...")
        
        req = urllib.request.Request(
            url,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        
        with urllib.request.urlopen(req) as response:
            with open(path, "wb") as f:
                f.write(response.read())
        
        print("Done!")
    else:
        print(f"{filename} already exists, skipping.")

nist_sp800_177r1.pdf already exists, skipping.
nist_sp800_61r2.pdf already exists, skipping.


In [2]:
#切割文本（chunking）
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 加载所有 PDF
all_docs = []
for filename in os.listdir("../data"):
    if filename.endswith(".pdf"):
        loader = PyPDFLoader(f"../data/{filename}")
        all_docs.extend(loader.load())

print(f"总页数: {len(all_docs)}")

# 切割成 chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
chunks = splitter.split_documents(all_docs)
print(f"总 chunks 数: {len(chunks)}")
print("\n--- 第一个 chunk 示例 ---")
print(chunks[0].page_content)

总页数: 208
总 chunks 数: 1288

--- 第一个 chunk 示例 ---
Date updated: April 3, 2025  
Withdrawn NIST Technical Series Publication 
 
 
Warning Notice 
 
The attached publication has been withdrawn (archived), and is provided solely for historical purposes. 
It may have been superseded by another publication (indicated below). 
 
Withdrawn Publication 
Series/Number NIST Special Publication (SP) 800-61 Revision 2 
Title Computer Security Incident Handling Guide 
Publication Date(s) August 2012 
Withdrawal Date April 3, 2025


In [3]:
# 生成 embedding 存入 FAISS
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("Building FAISS index...")
vectorstore = FAISS.from_documents(chunks, embeddings)

vectorstore.save_local("../data/faiss_index")
print("Index saved!")

Loading embedding model...


/var/folders/tk/x43kyqbn33v_ttx5rckk29cm0000gn/T/ipykernel_69987/286027542.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Building FAISS index...
Index saved!


In [4]:
# 测试检索
query = "How should an organization respond to a phishing email?"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
results = retriever.invoke(query)

print(f"Query: {query}\n")
for i, doc in enumerate(results):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)
    print(f"Source: {doc.metadata.get('source', 'unknown')}, Page: {doc.metadata.get('page', '?')}")
    print()

Query: How should an organization respond to a phishing email?

--- Chunk 1 ---
access to credit cards and bank accounts of the victim etc. Adversaries use a variety of tactics to 
make the recipient of the email believe that they have received the phishing email from a 
legitimate user or a legitimate domain, including: 
• Using a message-From: address that looks very close to one of the legitimate addresses 
the user is familiar with or from someone claiming to be an authority (IT administrator,
Source: ../data/nist_sp800_177r1.pdf, Page: 30

--- Chunk 2 ---
NIST SP 800-177 REV. 1  TRUSTWORTHY EMAIL 
 
 22 
This publication is available free of charge from: https://doi.org/10.6028/NIST.SP.800-177r1 
 
for physical harm or for committing ﬁnancial fraud) is called phishing. From the above 
discussion of email bombing attacks, it should be clear that spam can sometimes be a type of 
email bombing.  
Protecting the email infrastructure against spam is a challenging problem. �is is due to